* Name: Anish Samuel
* Course: DSC 670 
* Assignment: Week 2 - Prompt Engineering Assignment — Problem #4


## 1. Approach and prompt-engineering rationale

I separated extraction from structuring because the two tasks have different objectives. The first prompt emphasizes faithful transcription and instructs the model not to calculate, correct, or invent values. The second prompt focuses on mapping the extracted information into a predictable data structure.

This separation makes errors easier to diagnose. If a value is wrong in the final JSON, I can compare it with the intermediate extraction and determine whether the problem occurred during document reading or during schema mapping.

The invoice contains potentially confusing information, including a $1,000 credit against a $6.00 subtotal, percentage values, an additional discount condition, and placeholder text. I therefore instruct the model to preserve source values rather than silently “fix” them.

I created an open api key in the Open API website that expires in 7 days. The open api key is input as from the visual code if not available, else it will use the key that is available.


In [45]:
import os
import json
import requests
import httpx2
from getpass import getpass
from pathlib import Path
from openai import OpenAI

INVOICE_URL = "https://content.bellevue.edu/cst/dsc/670/dsc-670-exercise-invoice.pdf"
PDF_PATH = Path("/Users/anishsamuel/dsc530/miniconda3/envs/book_env/DSC 670/Week 2/dsc-670-exercise-invoice.pdf")

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Set it in the environment or enter it when prompted."
    )

http_client = httpx2.Client(headers={"Accept-Encoding": "gzip, deflate"})
client = OpenAI(http_client=http_client)
print("OpenAI client initialized.")

OpenAI client initialized.


I am checking if the pdf can be downloaded directly from the url. If url is not working, I have pointed to the downloaded file in my local.

In [46]:
if PDF_PATH.exists():
    if not PDF_PATH.read_bytes().startswith(b"%PDF-"):
        raise RuntimeError(
            f"The local file {PDF_PATH} is not a valid PDF. "
            "Replace it with the downloaded invoice PDF and rerun this cell."
        )
    print(f"Using local PDF: {PDF_PATH} ({PDF_PATH.stat().st_size:,} bytes).")
else:
    response = requests.get(INVOICE_URL, timeout=30)
    response.raise_for_status()

    if not response.content.startswith(b"%PDF-"):
        content_type = response.headers.get("content-type", "unknown")
        raise RuntimeError(
            f"The invoice URL returned {content_type}, not a PDF. "
            f"Save the invoice PDF as {PDF_PATH} and rerun this cell."
        )

    PDF_PATH.write_bytes(response.content)
    print(f"Downloaded {PDF_PATH} ({PDF_PATH.stat().st_size:,} bytes).")

Using local PDF: /Users/anishsamuel/dsc530/miniconda3/envs/book_env/DSC 670/Week 2/dsc-670-exercise-invoice.pdf (260,587 bytes).


## 2. Extract the invoice text with an LLM

I use the PDF itself as the model input so that the model can use the document's visual/layout context. The prompt uses explicit constraints: transcribe all visible information, preserve numeric values, do not calculate or correct anything, and return only the extracted text.

These constraints are important because extraction should preserve the source rather than turn into an accounting-validation task.


In [47]:
if not PDF_PATH.exists() or not PDF_PATH.read_bytes().startswith(b"%PDF-"):
    raise RuntimeError(
        f"{PDF_PATH} is not a valid PDF. Run the download cell and provide the authenticated invoice PDF."
    )

with PDF_PATH.open("rb") as f:
    uploaded_file = client.files.create(file=f, purpose="user_data")

print("Uploaded file:", uploaded_file.id)

Uploaded file: file-3Qmmaotu7QjbQPda3YXEc4


In [48]:
extraction_prompt = """
Extract the text from this invoice accurately and completely.

Rules:
1. Transcribe all visible invoice information, including the company/header,
   invoice date and number, contact information, purchase-order number,
   line items, quantities, descriptions, unit prices, amounts, discounts,
   subtotal, credit, tax rate, additional discount, balance due, bill-to
   information, and footer text.
2. Preserve numbers, currency amounts, percentages, names, addresses,
   and placeholder text exactly as represented in the invoice.
3. Do not calculate, correct, reinterpret, or fix any value.
4. Do not invent missing information.
5. Return only the extracted/transcribed text in a readable form.
"""

extraction_response = client.responses.create(
    model="gpt-5-mini",
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": extraction_prompt},
            {"type": "input_file", "file_id": uploaded_file.id},
        ],
    }],
)

extracted_text = extraction_response.output_text
print(extracted_text)


321 Avenue A Date: 6/28/2024
Portland, OR 12345 Invoice # 1111
Phone: (206) 555-1163 For PO # 123456
Fax: (206) 555-1164
someone@websitegoeshere.com
Quantity Description Unit price Amount Discount applied
1 Item Number 1 $ 2.00 2.00 $ 
1 Item Number 2 $ 2.00 2.00 $ 
1 Item Number 3 $ 2.00 2.00 $ 
$ -
$ -
$ -
$ -
$ -
$ -
$ -
$ -
Subtotal 6.00 $ 
Credit $ 1,000.00
Tax 9.80%
Additional discount 12%
Balance due (994.20) $ 
Company Name
INVOICE
If you have any questions concerning this invoice, contact 
<Name> at <phone or email>.
Make all checks payable to <Company name>.
Thank you for your business!
Bill To:
Natasha Jones
Central Beauty
123 Main St.
% discount 10%
Manhattan, NY 98765
(321) 555-1234
Items over this amount qualify for an additional discount $100


## 3. Summary of  the extraction result

The expected extraction should preserve the invoice number, date, three line items, subtotal, credit, tax rate, additional discount, balance due, customer information, and footer instructions.

The invoice's financial presentation is unusual: it shows a $1,000 credit against a $6.00 subtotal and a displayed balance due of $(994.20). Because this stage is extraction rather than financial validation, the model should preserve the displayed values. This demonstrates an important prompt-engineering principle: extraction and interpretation should not be silently combined.


## 4. Convert the extracted text to structured JSON

The second stage converts the transcription into machine-readable data. I use a JSON Schema rather than simply asking for “JSON.” The schema makes the required fields explicit and allows the response to be validated programmatically.

For information that is not explicitly provided, the prompt tells the model to use `null` rather than inventing a value.


In [49]:
invoice_schema = {
    "type": "object",
    "properties": {
        "vendor": {
            "type": "object",
            "properties": {
                "company_name": {"type": ["string", "null"]},
                "address": {"type": ["string", "null"]},
                "phone": {"type": ["string", "null"]},
                "fax": {"type": ["string", "null"]},
                "email": {"type": ["string", "null"]}
            },
            "required": ["company_name", "address", "phone", "fax", "email"],
            "additionalProperties": False
        },
        "invoice": {
            "type": "object",
            "properties": {
                "date": {"type": ["string", "null"]},
                "invoice_number": {"type": ["string", "null"]},
                "purchase_order_number": {"type": ["string", "null"]}
            },
            "required": ["date", "invoice_number", "purchase_order_number"],
            "additionalProperties": False
        },
        "bill_to": {
            "type": "object",
            "properties": {
                "name": {"type": ["string", "null"]},
                "company": {"type": ["string", "null"]},
                "address": {"type": ["string", "null"]},
                "city_state_zip": {"type": ["string", "null"]},
                "phone": {"type": ["string", "null"]},
                "discount_percent": {"type": ["number", "null"]}
            },
            "required": ["name", "company", "address", "city_state_zip",
                         "phone", "discount_percent"],
            "additionalProperties": False
        },
        "items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "quantity": {"type": ["number", "null"]},
                    "description": {"type": ["string", "null"]},
                    "unit_price": {"type": ["number", "null"]},
                    "amount": {"type": ["number", "null"]},
                    "discount_applied": {"type": ["string", "null"]}
                },
                "required": ["quantity", "description", "unit_price",
                             "amount", "discount_applied"],
                "additionalProperties": False
            }
        },
        "totals": {
            "type": "object",
            "properties": {
                "subtotal": {"type": ["number", "null"]},
                "credit": {"type": ["number", "null"]},
                "tax_rate_percent": {"type": ["number", "null"]},
                "additional_discount_percent": {"type": ["number", "null"]},
                "balance_due": {"type": ["number", "null"]}
            },
            "required": ["subtotal", "credit", "tax_rate_percent",
                         "additional_discount_percent", "balance_due"],
            "additionalProperties": False
        },
        "discount_condition": {
            "type": "object",
            "properties": {
                "threshold_amount": {"type": ["number", "null"]},
                "description": {"type": ["string", "null"]}
            },
            "required": ["threshold_amount", "description"],
            "additionalProperties": False
        },
        "footer": {
            "type": "object",
            "properties": {
                "contact_instruction": {"type": ["string", "null"]},
                "payment_instruction": {"type": ["string", "null"]},
                "thank_you": {"type": ["string", "null"]}
            },
            "required": ["contact_instruction", "payment_instruction", "thank_you"],
            "additionalProperties": False
        }
    },
    "required": ["vendor", "invoice", "bill_to", "items", "totals",
                 "discount_condition", "footer"],
    "additionalProperties": False
}


In [50]:
json_prompt = f"""
Convert the following invoice transcription into structured JSON.

Rules:
1. Use only information present in the transcription.
2. Do not invent, correct, or recalculate financial values.
3. Preserve the displayed balance due even if it appears unusual.
4. Convert numeric quantities, prices, amounts, percentages, and thresholds
   to JSON numbers when the source clearly provides them.
5. Use null when a requested field is not explicitly available.
6. Preserve placeholder text when it is actually printed in the invoice.
7. Return data that conforms exactly to the supplied JSON schema.

INVOICE TRANSCRIPTION:
{extracted_text}
"""

structured_response = client.responses.create(
    model="gpt-5-mini",
    input=json_prompt,
    text={
        "format": {
            "type": "json_schema",
            "name": "invoice",
            "strict": True,
            "schema": invoice_schema
        }
    }
)

invoice_json = json.loads(structured_response.output_text)
print(json.dumps(invoice_json, indent=2))


{
  "vendor": {
    "company_name": "Company Name",
    "address": "321 Avenue A\nPortland, OR 12345",
    "phone": "(206) 555-1163",
    "fax": "(206) 555-1164",
    "email": "someone@websitegoeshere.com"
  },
  "invoice": {
    "date": "6/28/2024",
    "invoice_number": "1111",
    "purchase_order_number": "123456"
  },
  "bill_to": {
    "name": "Natasha Jones",
    "company": "Central Beauty",
    "address": "123 Main St.",
    "city_state_zip": "Manhattan, NY 98765",
    "phone": "(321) 555-1234",
    "discount_percent": 10
  },
  "items": [
    {
      "quantity": 1,
      "description": "Item Number 1",
      "unit_price": 2.0,
      "amount": 2.0,
      "discount_applied": "$"
    },
    {
      "quantity": 1,
      "description": "Item Number 2",
      "unit_price": 2.0,
      "amount": 2.0,
      "discount_applied": "$"
    },
    {
      "quantity": 1,
      "description": "Item Number 3",
      "unit_price": 2.0,
      "amount": 2.0,
      "discount_applied": "$"
    }
  ],

## 5. Evaluation of the structured result

I would evaluate the output using three criteria: **completeness, fidelity, and structure**.

**Completeness** means important invoice fields are represented. **Fidelity** means source values such as the $6.00 subtotal, $1,000.00 credit, 9.80% tax rate, 12% additional discount, and displayed balance due of $(994.20) are not silently changed. **Structure** means the response is valid JSON and follows the schema.

The most important limitation is that structured extraction does not guarantee accounting correctness. The model can faithfully reproduce an unusual value without determining whether it is mathematically or commercially correct. That is appropriate for this exercise because extraction and validation are separate tasks.

For production use, I would add deterministic validation after extraction. I could compare line-item totals with the subtotal, verify required fields, flag unusual credits or negative balances, and route exceptions to a human reviewer. This combines LLM flexibility with conventional data-quality controls.


In [51]:
required_top_level = [
    "vendor", "invoice", "bill_to", "items", "totals",
    "discount_condition", "footer"
]

missing = [key for key in required_top_level if key not in invoice_json]

print("Top-level fields present:", not missing)
if missing:
    print("Missing:", missing)

print("Number of line items:", len(invoice_json["items"]))
print("Subtotal:", invoice_json["totals"]["subtotal"])
print("Balance due:", invoice_json["totals"]["balance_due"])


Top-level fields present: True
Number of line items: 3
Subtotal: 6.0
Balance due: -994.2


## 6. Conclusion

This exercise demonstrates why prompt engineering matters even when a model can understand a document. A vague request such as “extract this invoice as JSON” leaves many decisions unspecified. By separating extraction from transformation, defining constraints, preserving source values, and specifying a schema, I make the desired behavior more explicit.

The two-stage workflow also provides a practical debugging strategy. If the final JSON is incorrect, I can inspect the intermediate transcription before changing the JSON prompt. In a real data-processing application, that separation would make the system easier to test, monitor, and improve.
